In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def plot_all(t_range, v_n, raster, dt, vr):
    # Population rate in Hz
    rate = raster.mean(axis=0) / dt

    fig, ax = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

    # --- Top panel: Vm cloud + one sample trace + spike markers ---
    tt = np.tile(t_range, v_n.shape[0])
    vv = v_n.reshape(-1)
    ax[0].scatter(tt, vv, s=1, c='k', alpha=0.02, linewidths=0)

    ex = 0
    ax[0].plot(t_range, v_n[ex], color='#E6862E', lw=2.0)
    spike_t = t_range[raster[ex].astype(bool)]
    ax[0].plot(spike_t, vr * np.ones_like(spike_t), 'ko', ms=3)

    ax[0].set_ylabel(r'$V_m$ (V)')
    ax[0].set_ylim(vr - 0.001, -0.049)

    # --- Middle panel: raster ---
    spike_i, spike_j = np.where(raster > 0)
    ax[1].plot(t_range[spike_j], spike_i, 'k.', ms=1)
    ax[1].set_ylabel('neuron')
    ax[1].set_ylim(0, v_n.shape[0])

    # --- Bottom panel: rate ---
    ax[2].plot(t_range, rate, color='#2C7FB8', lw=1.8)
    ax[2].set_ylabel('rate (Hz)')
    ax[2].set_xlabel('time (s)')

    for a in ax:
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


# @markdown Execute this cell to enable the demo

t_max = 150e-3  #second
dt = 1e-3       #second
tau = 20e-3     #second
el = -60e-3     #milivolt
vr = -70e-3     #milivolt
vth = -50e-3    #milivolt
r = 100e6       #ohm
i_mean = 25e-11 #ampere

print(t_max, dt, tau, el, vr, vth, r, i_mean)

def random_ref_period(mu, sigma):
  # set random number generator
  np.random.seed(2020)

  # initialize step_end, t_range, n, v_n, syn and raster
  t_range = np.arange(0, t_max, dt)
  step_end = len(t_range)
  n = 500
  v_n = el * np.ones([n,step_end])
  syn = i_mean * (1 + 0.1*(t_max/dt)**(0.5)*(2*np.random.random([n,step_end]) - 1))
  raster = np.zeros([n,step_end])

  # initialize t_ref and last_spike
  t_ref = mu + sigma*np.random.normal(size=n)
  t_ref[t_ref<0] = 0
  last_spike = -t_ref * np.ones([n])

  # loop time steps
  for step, t in enumerate(t_range):
    if step==0:
      continue

    v_n[:,step] = v_n[:,step-1] + dt/tau * (el - v_n[:,step-1] + r*syn[:,step])

    # boolean array spiked indexes neurons with v>=vth
    spiked = (v_n[:,step] >= vth)
    v_n[spiked,step] = vr
    raster[spiked,step] = 1

    # boolean array clamped indexes refractory neurons
    clamped = (last_spike + t_ref > t)
    v_n[clamped,step] = vr
    last_spike[spiked] = t

  # plot multiple realizations of Vm, spikes and mean spike rate
  plot_all(t_range, v_n, raster, dt, vr)

  # plot histogram of t_ref
  plt.figure(figsize=(8,4))
  plt.hist(t_ref, bins=32, histtype='stepfilled', linewidth=0, color='C1')
  plt.xlabel(r'$t_{ref}$ (s)')
  plt.ylabel('count')
  plt.tight_layout()

widgets.interact(
         random_ref_period,
         mu=(0.01, 0.05, 0.01),
         sigma=(0.001, 0.02, 0.001),
)

0.15 0.001 0.02 -0.06 -0.07 -0.05 100000000.0 2.5e-10


interactive(children=(FloatSlider(value=0.02, description='mu', max=0.05, min=0.01, step=0.01), FloatSlider(va…

<function __main__.random_ref_period(mu, sigma)>